In [26]:
from torch.utils.data import Dataset,DataLoader
import pandas as pd 
import os
import cv2
import torch
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import numpy as np
from skimage.metrics import structural_similarity as ssim_fn

In [41]:
#dataclass for dependency injection



class TrainDataset(Dataset):
    def __init__(self,dir_path,csv_path,transformation=None,augment=False):
        self.prefix = "face-"
        self.dir = dir_path
        self.df = pd.read_csv(csv_path)
        self.transform = transformation
        self.augment = augment

        transforms = []
        if self.augment:
            print("applying special augments...")
            transforms += [
        #         # PRIMARY: random crop (75-100% of image) then resize back to 64x64
        #         # This is the main anti-averaging augmentation.
        #         # scale=(0.75, 1.0) means minimum crop is 75% of original area —
        #         # floor chosen so the face always remains visible after crop.
        #         # A.RandomResizedCrop(size=(64,64), scale=(0.75, 1.0), p=0.7),
                #  A.RandomCrop(height=48, width=48, p=0.7),  # 48 = 75% of 64, face still visible
                #  A.Resize(height=64, width=64),  
        #         # Combines shift, scale, rotate in one transform.
        #         # shift_limit=0.1  : translate up to 10% of image width/height
        #         # scale_limit=0.2  : zoom in/out by up to 20%
        #         # rotate_limit=25  : rotate up to ±25° — heavier but still natural
        #         # A.ShiftScaleRotate(
        #         #     shift_limit=0.1,
        #         #     scale_limit=0.2,
        #         #     rotate_limit=25,
        #         #     border_mode=cv2.BORDER_REFLECT_101,
        #         #     p=0.6
        #         # ),

                A.Affine(
                translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
                scale=(0.95, 1.05),
                rotate=(-10, 10),
                border_mode=cv2.BORDER_REFLECT_101,
                p=0.6
                ),
                # Horizontal flip — natural for faces, effectively doubles dataset
                A.HorizontalFlip(p=0.5),
 
                # Gaussian noise — forces robust feature learning
                # var_limit=(5,20): mild enough that face structure is preserved
                # A.GaussNoise(var_limit=(5.0, 20.0), p=0.3),
                # A.GaussNoise(std_range=(0.02, 0.09), p=0.3),
               
 
                # # Heavier lighting variation than before (0.2 vs 0.1 previously)
                # A.RandomBrightnessContrast(
                #     brightness_limit=0.2,
                #     contrast_limit=0.2,
                #     p=0.4
                # ),
                A.RandomBrightnessContrast(
                brightness_limit=0.15,
                contrast_limit=0.15,
                p=0.3
            ),

                A.GaussNoise(std_range=(0.01, 0.05), p=0.2),
            ]
 
        transforms += [
            A.Normalize(mean=[0.0, 0.0, 0.0], std=[1.0, 1.0, 1.0],
                        max_pixel_value=255.0),
            ToTensorV2()
        ]
        self.transforms = A.Compose(transforms)
 
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        label = int(self.df.iloc[index]["glasses"])
        img_name = self.prefix + f"{index + 1}.png"
        img_path = os.path.join(self.dir, img_name)

        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # normalize to [0,1]
        # img = img / 255.0
       
        # img = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1)
        # img = self.transform(img) if self.transform else img
        img = self.transforms(image=img)["image"]
        # img = img.numpy()
        return img,label
    

#dataloader


dataset = TrainDataset(
    dir_path="resized_dataset",
    csv_path="train-corrected.csv",
    augment=True
)
# dataset.augment = True
loader = DataLoader(dataset, batch_size=64, shuffle=True,pin_memory=True)

def visualize_augments(loader,n=16):
    # dataset.augment = True
    imgs,labels = next(iter(loader))

    imgs         = imgs[:n].permute(0, 2, 3, 1).numpy()
    labels       = labels[:n].numpy()

    label_names={
        1:"glasses",
        0:"no-glasses"
    }
    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    for i, ax in enumerate(axes.flat):
        ax.imshow(np.clip(imgs[i], 0, 1))
        ax.set_title(label_names[labels[i]], fontsize=8)
        ax.axis("off")
    plt.suptitle("augments visualized")
    plt.tight_layout()
    plt.show()
    # dataset.augment = False
    



applying special augments...


In [28]:
x, y = next(iter(loader))

print(x.min(), x.max())
print(x.shape)
print(y[:10])

tensor(0.) tensor(0.9961)
torch.Size([64, 3, 64, 64])
tensor([1, 1, 1, 0, 0, 0, 0, 1, 1, 0])


In [47]:
# #VAE class

import torch
import torch.nn as nn
import torch.nn.functional as F

# class VAE(nn.Module):
#     def __init__(self, z_dim=128):
#         super().__init__()

#         # Encoder
#         self.encoder = nn.Sequential(
#             nn.Conv2d(3, 32, 4, 2, 1),   # 64 → 32
#             nn.LeakyReLU(),
#             nn.Conv2d(32, 64, 4, 2, 1),  # 32 → 16
#             nn.LeakyReLU(),
#             nn.Conv2d(64, 128, 4, 2, 1), # 16 → 8
#             nn.LeakyReLU(),
#         )

#         self.fc_mu = nn.Linear(128 * 8 * 8, z_dim)
#         self.fc_logvar = nn.Linear(128 * 8 * 8, z_dim)

#         # Decoder
#         self.fc_dec = nn.Linear(z_dim, 128 * 8 * 8)

#         self.decoder = nn.Sequential(
#             nn.ConvTranspose2d(128, 64, 4, 2, 1),  # 8 → 16
#             nn.LeakyReLU(),
#             nn.ConvTranspose2d(64, 32, 4, 2, 1),   # 16 → 32
#             nn.LeakyReLU(),
#             nn.ConvTranspose2d(32, 3, 4, 2, 1),    # 32 → 64
#             nn.Sigmoid()
#         )

#     def encode(self, x):
#         x = self.encoder(x)
#         x = x.view(x.size(0), -1)
#         mu = self.fc_mu(x)
#         logvar = self.fc_logvar(x)
#         return mu, logvar

#     def reparameterize(self, mu, logvar):
#         std = torch.exp(0.5 * logvar)
#         eps = torch.randn_like(std)
#         return mu + eps * std

#     def decode(self, z):
#         x = self.fc_dec(z)
#         x = x.view(-1, 128, 8, 8)
#         return self.decoder(x)

#     def forward(self, x):
#         mu, logvar = self.encode(x)
#         z = self.reparameterize(mu, logvar)
#         recon = self.decode(z)
#         return recon, mu, logvar



# def vae_loss(recon_x, x, mu, logvar, beta=0.05):
#     recon_loss = F.binary_cross_entropy(recon_x, x, reduction='mean')

#     kl = -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

#     return recon_loss + beta * kl


class VAE(nn.Module):
    def __init__(self, z_dim=128):
        super().__init__()
        self.z_dim = z_dim
        self.label_emb = nn.Embedding(2, 16)

        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 4, 2, 1),
            nn.ReLU(),
    

            nn.Conv2d(32, 64, 4, 2, 1),
            nn.ReLU(),
            

            nn.Conv2d(64, 128, 4, 2, 1),
            nn.ReLU()
        )

        self.fc_mu = nn.Linear(128*8*8 + 16, z_dim)
        self.fc_logvar = nn.Linear(128*8*8 + 16, z_dim)

        self.fc_dec = nn.Linear(z_dim + 16, 128*8*8)

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, 2, 1),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.ConvTranspose2d(64, 32, 4, 2, 1),
            nn.ReLU(),
            nn.Dropout(0.2),
            
            nn.ConvTranspose2d(32, 3, 4, 2, 1),
            nn.Sigmoid()
        )

    def encode(self, x, y):
        h = self.encoder(x)
        h = h.view(h.size(0), -1)
        y_emb = self.label_emb(y)
        h = torch.cat([h, y_emb], dim=1)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z, y):
        y_emb = self.label_emb(y)
        z = torch.cat([z, y_emb], dim=1)
        h = self.fc_dec(z)
        h = h.view(-1, 128, 8, 8)
        return self.decoder(h)

    def forward(self, x, y):
        mu, logvar = self.encode(x, y)
        z = self.reparameterize(mu, logvar)
        recon = self.decode(z, y)
        return recon, mu, logvar


def vae_loss(recon, x, mu, logvar, beta):
    recon_loss = F.mse_loss(recon, x)
    kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())/x.size(0)
    return recon_loss + beta * kl




In [ ]:
def class_level_stats(loader,model,device):
    model.eval()
    class_level_mu,class_level_std = {},{}
    mus = {
        0:[],
        1:[],
    }
    with torch.no_grad():
        for img,label in loader:
            img = img.to(device)
            mu,_ = model.encode(img)
            for i,lbl in enumerate(label):
                mus[lbl.item()].append(mu[i].cpu())

    
    for i in range(2):
        stacked = torch.stack(mus[i])
        class_level_mu[i] = stacked.mean(dim=0)
        class_level_std[i] = stacked.std(dim=0)

    return class_level_mu,class_level_std


# def conditional_sampling(model,loader,device,label,mus,stds,n_samples=3,temp=1.2):
    
#     # mus,stds = class_level_stats(loader,model,device)
#     mu = mus[label].to(device)
#     std = stds[label].to(device)
#     with torch.no_grad():
#         eps = torch.randn(n_samples,mu.shape[0],device=device)
#         z = mu + temp*std*eps
#         imgs = model.decode(z)
    
#     imgs = imgs.cpu()
#     return imgs



def visualize_generated(imgs_glasses, imgs_noglasses):
    all_imgs    = torch.cat([imgs_glasses, imgs_noglasses], dim=0)
    all_imgs_np = all_imgs.permute(0, 2, 3, 1).numpy()
    titles      = ["glasses"] * 3 + ["no glasses"] * 3
 
    fig, axes = plt.subplots(1, 6, figsize=(15, 3))
    for i, ax in enumerate(axes):
        ax.imshow(np.clip(all_imgs_np[i], 0, 1))
        ax.set_title(titles[i])
        ax.axis("off")
    plt.suptitle("Conditionally Generated Faces (VAE)")
    plt.tight_layout()
    plt.show()






In [29]:
def compute_ssim(generated, real):
    """
    Mean SSIM across N (generated, real) image pairs.
 
    Args:
        generated : (N, C, H, W) float tensor [0,1]
        real      : (N, C, H, W) float tensor [0,1]
    Returns:
        float — mean SSIM  (1.0 = identical, 0.0 = no similarity)
    """
    scores  = []
    gen_np  = generated.cpu().numpy()
    real_np = real.cpu().numpy()
 
    for g, r in zip(gen_np, real_np):
        g     = np.clip(g.transpose(1, 2, 0), 0, 1)  # CHW → HWC
        r     = np.clip(r.transpose(1, 2, 0), 0, 1)
        score = ssim_fn(g, r, channel_axis=2, data_range=1.0)
        scores.append(score)
 
    return float(np.mean(scores))


def ssim_eval(imgs_glasses,imgs_noglasses):
    real_imgs, _ = next(iter(loader))
    real_sample   = real_imgs[:6]
    all_generated = torch.cat([imgs_glasses, imgs_noglasses], dim=0)
 
    ssim_score = compute_ssim(all_generated, real_sample)
    print("ssim-score: ",ssim_score)


In [ ]:
#train VAE
# print(device)
def train_vae(z_dim=64,beta=0.05,activation="leakyRELU",lr=1e-3,epochs=80,device="cpu",temp=1.0):
    loss_type = "MSE"
    # activation="leakyRELU"
    # z_dim=64
    # beta=0.05
    model = VAE(z_dim=z_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5,
    min_lr=1e-5,
    verbose=False
)
    os.makedirs(f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}",exist_ok=True)
    visualize_augments(loader)
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        beta_opti = max(0.00001,min(beta,(epoch/epochs)*4*beta))
        for imgs,labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            recon, mu, logvar = model(imgs,labels)
            loss = vae_loss(recon, imgs, mu, logvar,beta_opti)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        
        # 16 images, z_dim=64
        # with torch.no_grad():
        #     z = torch.randn(1, z_dim).to(device)*temp 
        #     y0 = torch.tensor([0]).to(device)
        #     y1 = torch.tensor([1]).to(device)
        #     generated0 = model.decode(z,y0)[0]
        #     generated1 = model.decode(z,y1)[0]
        
        # generated0 = generated0.cpu().permute(1,2,0).numpy()*255
        # generated1 = generated1.cpu().permute(1,2,0).numpy()*255

        # generated0 = generated0.astype("uint8")
        # generated1 = generated1.astype("uint8")

        # combined = np.concat([generated0,generated1],axis=1)

        # img = cv2.cvtColor(combined,cv2.COLOR_RGB2BGR)
        # # save = cv2.resize(img,(256,256),interpolation=cv2.INTER_NEAREST)
        # # save_image(generated,f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/VAE_epoch_{epoch}.png")
        # cv2.imwrite(f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/VAE_epoch_{epoch}.png",img)
        
        
        # save_image(save,f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/VAE_epoch_{epoch}.png")


        print(f"Epoch {epoch}, Loss: {total_loss}")
    # class_mu,class_std = class_level_stats(loader,model,device)
        avg_loss = total_loss/len(loader)
        scheduler.step(avg_loss)
    # torch.save({
    #     "model_state":model.state_dict(),
    #     "z_dim":z_dim,
    #     "lr":lr
    # },
    # f"generated/VAE/z_size={z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}.pt")
    return model,lr,loss_type,beta,activation

In [32]:
def conditional_sampling(model,device,label,n=3):
    model.eval()
    with torch.no_grad():
        z = torch.randn(n, model.z_dim).to(device)
        y = torch.full((n,), label, dtype=torch.long).to(device)
        imgs = model.decode(z, y)
        return imgs.cpu()

In [ ]:
import torchvision.utils as vutils
def main():
    torch.set_float32_matmul_precision('high')
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model,lr,loss_type,beta,activation = train_vae()
    # class_level_stats(loader,model,device)
    glasses,no_glasses = conditional_sampling(model,device,1),conditional_sampling(model,device,0)
    visualize_generated(glasses,no_glasses)
    for i in range(3):
        vutils.save_image(glasses[i],f"generated/VAE/z_size={model.z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/conditional_glasses_{i+1}.png")
        vutils.save_image(no_glasses[i],f"generated/VAE/z_size={model.z_dim}-lr={lr}-loss={loss_type}-beta={beta}-activation={activation}/conditional_no_glasses_{i+1}.png")
    # vutils.save_image(glasses,"generated/VAE/glasses.png",nrows=3)
    # vutils.save_image(no_glasses,"generated/VAE/no-glasses.png",nrows=3)
    # ssim_eval(glasses,no_glasses)

if __name__ == "__main__":
    main()